In [7]:
import os
import sys
from datasets import load_dataset, Dataset
from dotenv import load_dotenv
from datetime import datetime
import pandas as pd

In [8]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU model: {torch.cuda.get_device_name(0)}")

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU model: NVIDIA GeForce RTX 3060 Laptop GPU


## LOAD DATA FROM MINIO

In [ ]:
# endpoint_url = os.getenv("ENDPOINT_URL")
# key = os.getenv("MINIO_ROOT_USER")
# secret = os.getenv("MINIO_ROOT_PASSWORD")

In [9]:
# def load_data_for_nlp(s3a_path):
#     print(f"[*] Đang kết nối tới MinIO tại {endpoint_url}...")
#     print(f"[*] Nạp dữ liệu từ: {s3a_path}")
#     try:
#         dataset = load_dataset(
#             "parquet",
#             data_files=s3a_path,
#             storage_options={
#                 "key": key,
#                 "secret": secret,
#                 "client_kwargs":{
#                     "endpoint_url": endpoint_url
#                 }
#             }
#         )
#         hf_dataset = dataset["train"]
#         print(f"Đã nạp xong Dataset!")
#         print(f"Tổng số bản ghi {len(hf_dataset)}")
#         print(f"Các cột dữ liệu: {hf_dataset.column_names}")
        
#         return hf_dataset
#     except Exception as e:
#         print(f"[!] Lỗi khi nạp dữ liệu trực tiếp: {e}")
#         return None

In [10]:
# # path = os.path.abspath(os.path.join(os.path.dirname(__file__), "../.."))
# # print(path)
# bucket_name = "silver"
# base_prefix = "nlp/dataset"
# s3_path = get_layer_path("s3://", bucket_name, base_prefix, "2026/03/30") 
# print(s3_path)
# dataset = load_data_for_nlp(s3_path + "*.parquet")
# if dataset:
#     # Kiểm tra thử 1 dòng để xem cấu trúc
#     print("\n[Mẫu dữ liệu dòng 0]:")
#     print(dataset[0])

In [11]:
# print(type(dataset))
# for i in range(5):
#     for key, values in dataset[i].items():
#         print(f"{key}: {values}")
#     print("\n")

In [12]:
df = pd.read_csv("output.csv")
print(len(df))
df_cleaned = df.dropna(subset=['rating', 'content'])
print(f"[*] Số lượng bản ghi còn lại sau khi lọc null: {len(df)}")

70078
[*] Số lượng bản ghi còn lại sau khi lọc null: 70078


In [13]:
dataset = Dataset.from_pandas(df_cleaned, preserve_index=False)
dataset = dataset.filter(lambda x: x['rating'] is not None and x['content'] is not None)
print(f"[*] Số lượng bản ghi còn lại sau khi lọc null: {len(dataset)}")

Filter: 100%|██████████| 67803/67803 [00:00<00:00, 144700.94 examples/s]

[*] Số lượng bản ghi còn lại sau khi lọc null: 67803


In [14]:
# Cell: Preprocessing
def clean_text(text):
    import re
    # Ép kiểu string để tránh lỗi "float object has no attribute lower" khi đọc từ CSV
    text = str(text).lower() 
    text = re.sub(r'<.*?>', '', text) # Xóa HTML
    text = re.sub(r'[^a-z0-9.,!?\s]', '', text) # Giữ dấu câu quan trọng
    return re.sub(r'\s+', ' ', text).strip()

def preprocess_fn(examples):
    # Xử lý text
    examples["content_clean"] = [clean_text(t) for t in examples["content"]]
    
    # Scale rating 1-10 về 0-1
    labels = []
    for r in examples["rating"]:
        try:
            val = (float(r) - 1.0) / 9.0
            labels.append(val)
        except (TypeError, ValueError):
            labels.append(0.5) 
            
    examples["label"] = labels
    return examples

# Áp dụng map trên dataset đã được chuyển đổi từ Pandas
dataset_cleaned = dataset.map(preprocess_fn, batched=True)

Map: 100%|██████████| 67803/67803 [00:05<00:00, 12547.09 examples/s]


In [15]:
from sklearn.model_selection import GroupShuffleSplit
import pandas as pd
from datasets import Dataset, DatasetDict

# 1. Chuyển Dataset sang Pandas để xử lý index theo Group
df = dataset_cleaned.to_pandas()

# 2. Khởi tạo bộ chia theo review_id (80% Train, 20% còn lại cho Val & Test)
gss = GroupShuffleSplit(n_splits=1, train_size=0.8, random_state=42)
train_idx, temp_idx = next(gss.split(df, groups=df['review_id']))

train_df = df.iloc[train_idx]
temp_df = df.iloc[temp_idx]

# 3. Chia 20% còn lại thành Val (10%) và Test (10%)
gss_val = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=42)
val_idx, test_idx = next(gss_val.split(temp_df, groups=temp_df['review_id']))

val_df = temp_df.iloc[val_idx]
test_df = temp_df.iloc[test_idx]

# 4. Đưa ngược về Hugging Face DatasetDict để tiện Tokenize
final_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df)
})

print(f"Kết quả chia tập dữ liệu:")
print(f"- Train: {len(final_dataset['train'])} dòng")
print(f"- Val:   {len(final_dataset['validation'])} dòng")
print(f"- Test:  {len(final_dataset['test'])} dòng")

Kết quả chia tập dữ liệu:
- Train: 54242 dòng
- Val:   6780 dòng
- Test:  6781 dòng


In [16]:
# Cell: Chuẩn bị dữ liệu cho RoBERTa
from transformers import AutoTokenizer

roberta_checkpoint = "roberta-base"
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_checkpoint)

def tokenize_roberta(examples):
    # Sử dụng tokenizer mới của RoBERTa
    return roberta_tokenizer(
        examples["content_clean"], 
        truncation=True, 
        padding="max_length", 
        max_length=256
    )

print("[*] Đang Tokenize dữ liệu cho RoBERTa...")
# Sử dụng lại final_dataset chứa các text đã làm sạch và được chia tách
roberta_tokenized_datasets = final_dataset.map(tokenize_roberta, batched=True)

# Dọn dẹp các cột thừa để rảnh RAM
roberta_tokenized_datasets = roberta_tokenized_datasets.remove_columns(["content", "content_clean", "review_id", "rating"])
roberta_tokenized_datasets.set_format("torch")

print("✅ Đã chuẩn bị xong dữ liệu cho RoBERTa!")

d:\chip\project\movie-review-sentiment\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\thung\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


[*] Đang Tokenize dữ liệu cho RoBERTa...


Map: 100%|██████████| 6781/6781 [00:00<00:00, 8111.02 examples/s]

✅ Đã chuẩn bị xong dữ liệu cho RoBERTa!


In [18]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # BERT Regression trả về mảng (batch_size, 1), cần đưa về mảng 1 chiều
    predictions = predictions.squeeze()
    
    mae = mean_absolute_error(labels, predictions)
    mse = mean_squared_error(labels, predictions)
    rmse = np.sqrt(mse)
    
    # Tính toán sai số thực tế trên thang điểm 10 để Chip dễ quan sát
    mae_scale_10 = mae * 9.0
    
    return {
        "mae": mae,
        "rmse": rmse,
        "mae_on_scale_10": mae_scale_10
    }

In [ ]:
# Cell: Huấn luyện RoBERTa
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

# 1. Load model RoBERTa cho Regression (num_labels=1)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_checkpoint, num_labels=1)

# 2. Cấu hình tham số (Đổi output_dir thành results_roberta)
roberta_training_args = TrainingArguments(
    output_dir="../models/results_roberta",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mae",
    greater_is_better=False,
    logging_steps=100,
    push_to_hub=False,
    report_to="none",
    fp16=True,                  # Vẫn bật fp16 để tận dụng tối đa GPU trên server
    dataloader_num_workers=4    
)

# 3. Khởi tạo Trainer mới
roberta_trainer = Trainer(
    model=roberta_model,
    args=roberta_training_args,
    train_dataset=roberta_tokenized_datasets["train"],
    eval_dataset=roberta_tokenized_datasets["validation"],
    processing_class=roberta_tokenizer,
    compute_metrics=compute_metrics, # Dùng lại hàm compute_metrics đã định nghĩa trước đó
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4926.33it/s]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
print(f"[*] Bắt đầu huấn luyện: {roberta_checkpoint}...")
roberta_trainer.train()

[*] Bắt đầu huấn luyện: roberta-base...


Epoch,Training Loss,Validation Loss,Mae,Rmse,Mae On Scale 10
1,0.025851,0.022293,0.102804,0.149308,0.925240
2,0.020141,0.021878,0.102363,0.147911,0.921269
3,0.014362,0.021939,0.100486,0.148119,0.904370


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.25it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

TrainOutput(global_step=10173, training_loss=0.022681248969623008, metrics={'train_runtime': 2270.872, 'train_samples_per_second': 71.658, 'train_steps_per_second': 4.48, 'total_flos': 2.1407312587908096e+16, 'train_loss': 0.022681248969623008, 'epoch': 3.0})

In [24]:
print("[*] Đang đánh giá RoBERTa trên tập Test...")
roberta_test_results = roberta_trainer.predict(roberta_tokenized_datasets["test"])

print("\nKẾT QUẢ CỦA ROBERTA:")
print(f"- MAE (0-1): {roberta_test_results.metrics['test_mae']:.4f}")
print(f"- RMSE (0-1): {roberta_test_results.metrics['test_rmse']:.4f}")
print(f"- Sai số trung bình: {roberta_test_results.metrics['test_mae_on_scale_10']:.2f} điểm")

[*] Đang đánh giá RoBERTa trên tập Test...



KẾT QUẢ CỦA ROBERTA:
- MAE (0-1): 0.1007
- RMSE (0-1): 0.1471
- Sai số trung bình: 0.91 điểm


In [25]:
roberta_model_path = "../models/roberta-movie-rating-regressor"
roberta_trainer.save_model(roberta_model_path)
roberta_tokenizer.save_pretrained(roberta_model_path)

print(f"✅ Đã lưu RoBERTa tại: {roberta_model_path}")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.15it/s]

✅ Đã lưu RoBERTa tại: ../models/roberta-movie-rating-regressor
